In [4]:
"""
    geosearch.py

    MediaWiki API Demos
    Demo of `Geosearch` module: Search for wiki pages nearby

    MIT License
"""

import requests

S = requests.Session()

URL = "https://en.wikipedia.org/w/api.php"

HEADERS = {
    "User-Agent": "TensorTours/1.0 (https://tensortours.com; contact@tensortours.com)"
}

PARAMS = {
    "format": "json",
    "list": "geosearch",
    "gscoord": "45.513176133896465|-122.67682430442373",
    "gslimit": "10",
    "gsradius": "10000",
    "action": "query"
}

R = S.get(url=URL, params=PARAMS, headers=HEADERS)
DATA = R.json()

PLACES = DATA['query']['geosearch']

for place in PLACES:
    print(place['title'])

Porter Portland
KOIN Tower
Portland Marriott Downtown Waterfront
Umpqua Holdings Corporation
Umpqua Bank Plaza
Jefferson Substation
River Legend
Edith Green – Wendell Wyatt Federal Building
Federal Reserve Bank of San Francisco Portland Branch
Keller Auditorium


In [8]:
"""
    Wikidata + Wikipedia + Wikimedia Commons
    
    Find a notable historical site in Portland, fetch all data including images
"""

import requests
from IPython.display import display, HTML, Image
import urllib.parse

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"

# Portland, Oregon coordinates (from cell above)
LAT = 45.513176133896465
LON = -122.67682430442373
RADIUS_KM = 10

HEADERS = {
    "User-Agent": "TensorTours/1.0 (https://tensortours.com; contact@tensortours.com)",
    "Accept": "application/sparql-results+json"
}

# Step 1: Find most notable historical site nearby
# Focusing on positive tourist-friendly types: landmarks, museums, monuments, historic buildings
# Excluding: cemeteries, funeral homes, memorials to death
QUERY = f"""
SELECT DISTINCT ?place ?placeLabel ?placeDescription ?location ?distance 
       ?image ?article ?inception
       (COUNT(DISTINCT ?sitelink) AS ?popularity) WHERE {{
  SERVICE wikibase:around {{
    ?place wdt:P625 ?location .
    bd:serviceParam wikibase:center "Point({LON} {LAT})"^^geo:wktLiteral .
    bd:serviceParam wikibase:radius "{RADIUS_KM}" .
    bd:serviceParam wikibase:distance ?distance .
  }}
  
  # Match tourist-friendly historical types
  VALUES ?type {{ 
    wd:Q2319498    # landmark
    wd:Q35112127   # historic landmark
    wd:Q33506      # museum
    wd:Q24354      # theater
    wd:Q41176      # building
    wd:Q12280      # bridge
    wd:Q16560      # palace
    wd:Q3947       # house
    wd:Q655686     # historic house
    wd:Q174782     # public square
    wd:Q11303      # skyscraper
    wd:Q483453     # fountain
  }}
  ?place wdt:P31/wdt:P279* ?type .
  
  # Exclude grim places
  FILTER NOT EXISTS {{ ?place wdt:P31/wdt:P279* wd:Q39614 . }}   # cemetery
  FILTER NOT EXISTS {{ ?place wdt:P31/wdt:P279* wd:Q1360568 . }} # funeral home
  FILTER NOT EXISTS {{ ?place wdt:P31/wdt:P279* wd:Q5003624 . }} # memorial (often war/death)
  FILTER NOT EXISTS {{ ?place wdt:P31/wdt:P279* wd:Q4830453 . }} # business establishment
  
  # Optional: get image, Wikipedia article, founding date
  OPTIONAL {{ ?place wdt:P18 ?image . }}
  OPTIONAL {{ ?place wdt:P571 ?inception . }}
  OPTIONAL {{
    ?article schema:about ?place ;
             schema:isPartOf <https://en.wikipedia.org/> .
  }}
  
  ?sitelink schema:about ?place .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
}}
GROUP BY ?place ?placeLabel ?placeDescription ?location ?distance ?image ?article ?inception
ORDER BY DESC(?popularity)
LIMIT 1
"""

print("🔍 Searching for notable landmarks near Portland, OR...\n")
response = requests.get(WIKIDATA_ENDPOINT, params={"query": QUERY}, headers=HEADERS)
data = response.json()

if not data["results"]["bindings"]:
    print("No sites found!")
else:
    result = data["results"]["bindings"][0]
    
    place_name = result["placeLabel"]["value"]
    place_id = result["place"]["value"].split("/")[-1]
    description = result.get("placeDescription", {}).get("value", "No description")
    distance = float(result["distance"]["value"])
    popularity = int(result["popularity"]["value"])
    coords = result["location"]["value"]
    inception = result.get("inception", {}).get("value", "Unknown")[:4] if "inception" in result else "Unknown"
    image_url = result.get("image", {}).get("value", None)
    wiki_article = result.get("article", {}).get("value", None)
    
    print(f"{'='*60}")
    print(f"📍 {place_name}")
    print(f"{'='*60}")
    print(f"Wikidata ID: {place_id}")
    print(f"Description: {description}")
    print(f"Founded: {inception}")
    print(f"Distance: {distance:.2f} km from center")
    print(f"Popularity: {popularity} sitelinks")
    print(f"Coordinates: {coords}")
    print()
    
    # Step 2: Fetch Wikipedia article content
    if wiki_article:
        wiki_title = urllib.parse.unquote(wiki_article.split("/wiki/")[-1])
        print(f"📖 Fetching Wikipedia article: {wiki_title}\n")
        
        wiki_params = {
            "action": "query",
            "format": "json",
            "titles": wiki_title,
            "prop": "extracts|pageimages|images|categories",
            "exintro": False,
            "explaintext": True,
            "piprop": "original",
            "imlimit": "20",
            "cllimit": "20"
        }
        
        wiki_response = requests.get(WIKIPEDIA_API, params=wiki_params, headers=HEADERS)
        wiki_data = wiki_response.json()
        
        pages = wiki_data["query"]["pages"]
        page = list(pages.values())[0]
        
        # Article content
        extract = page.get("extract", "No content available")
        print(f"{'='*60}")
        print("📜 WIKIPEDIA CONTENT")
        print(f"{'='*60}")
        print(extract[:3000] + "..." if len(extract) > 3000 else extract)
        print()
        
        # Categories
        categories = page.get("categories", [])
        if categories:
            print(f"{'='*60}")
            print("🏷️ CATEGORIES")
            print(f"{'='*60}")
            for cat in categories[:10]:
                print(f"  • {cat['title'].replace('Category:', '')}")
            print()
        
        # Get all images from the article
        images = page.get("images", [])
        image_files = [img["title"] for img in images if not any(x in img["title"].lower() for x in ["icon", "logo", "flag", "commons-logo", "edit", "lock"])]
        
    else:
        wiki_title = place_name
        image_files = []
    
    # Step 3: Fetch images from Wikimedia Commons
    print(f"{'='*60}")
    print("🖼️ IMAGES")
    print(f"{'='*60}")
    
    # Get main Wikidata image
    if image_url:
        print(f"\n📷 Main image from Wikidata:")
        print(f"   {image_url}")
        display(Image(url=image_url, width=500))
    
    # Get additional images from Wikipedia article
    if image_files:
        print(f"\n📷 Additional images from Wikipedia ({len(image_files)} found):")
        
        # Fetch actual URLs for first 5 images
        for img_title in image_files[:15]:
            img_params = {
                "action": "query",
                "format": "json",
                "titles": img_title,
                "prop": "imageinfo",
                "iiprop": "url|extmetadata",
                "iiurlwidth": "800"
            }
            img_response = requests.get(WIKIPEDIA_API, params=img_params, headers=HEADERS)
            img_data = img_response.json()
            
            img_pages = img_data["query"]["pages"]
            img_page = list(img_pages.values())[0]
            
            if "imageinfo" in img_page:
                img_info = img_page["imageinfo"][0]
                thumb_url = img_info.get("thumburl", img_info.get("url"))
                
                # Get image description if available
                metadata = img_info.get("extmetadata", {})
                img_desc = metadata.get("ImageDescription", {}).get("value", "")
                if img_desc:
                    img_desc = img_desc[:100].replace("<", "").replace(">", "")
                
                print(f"\n   • {img_title.replace('File:', '')}")
                if img_desc:
                    print(f"     {img_desc}...")
                display(Image(url=thumb_url, width=400))
    
    print(f"\n{'='*60}")
    print("✅ Data retrieval complete!")
    print(f"{'='*60}")

🔍 Searching for notable landmarks near Portland, OR...

📍 Providence Park
Wikidata ID: Q262291
Description: sports stadium in Portland, Oregon, United States
Founded: 1926
Distance: 1.47 km from center
Popularity: 22 sitelinks
Coordinates: Point(-122.691666666 45.521388888)

📖 Fetching Wikipedia article: Providence_Park

📜 WIKIPEDIA CONTENT
Providence Park (formerly Jeld-Wen Field; PGE Park; Civic Stadium; originally Multnomah Stadium; and from 1893 until the stadium was built, Multnomah Field) is an outdoor soccer venue located in the Goose Hollow neighborhood of Portland, Oregon. It is the home of the Portland Timbers of Major League Soccer (MLS) and Portland Thorns FC of the National Women's Soccer League (NWSL). Providence Park is currently the oldest facility to be configured as a soccer-specific stadium for use by an MLS team, and is one of the most historic grounds used by any United States professional soccer team. It has existed in rudimentary form since 1893, and as a complet


📷 Additional images from Wikipedia (3 found):

   • Aiga bus trans.svg
     This image is from the a rel="nofollow" class="external text" href="http://www.aiga.org/symbol-sign...



   • Civic Stadium, 1974.JPG
     Civic Stadium, now Providence Park, in Portland, Oregon...



   • Exterior of Providence Park from south in 2024 - Portland, OR.jpg
     The exterior of a href="https://en.wikipedia.org/wiki/Providence_Park" class="extiw" title="en:Prov...



✅ Data retrieval complete!
